# 🏋️ Day 1 실습 — 로컬 개발환경 & 랭체인 첫 호출

📖 **강의 연계**: Day 1 강의교안 **모듈 1-6** "랭체인 개요 & 첫 호출" + **모듈 1-7** "Colab 이식 + 마이 서비스 조각 선언"

✅ **완료 기준**
- [ ] `llm.invoke()`로 LLM 응답을 받고 `response.content`를 출력한다
- [ ] `SystemMessage` / `HumanMessage`를 분리해 v2를 실행한다
- [ ] 마이 서비스 조각의 첫 호출 코드를 작성하고 LangSmith 링크를 제출한다

⚠️ **우측 상단 커널이 `.venv`인지 확인하세요** (Colab 아님)
`Ctrl+Shift+P` → "Python: Select Interpreter" → `.venv` 선택 후 우측 상단 커널 이름 확인

In [4]:
# 셀 2 · 환경 점검 — 반드시 가장 먼저 실행하세요
from dotenv import load_dotenv
import os

load_dotenv()  # .env 파일 내용을 환경변수로 등록

# OpenAI API 키 존재 확인 — 없으면 여기서 멈추고 안내
assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY 없음 — .env 파일을 확인하세요\n"
    "해결: 1) 프로젝트 루트에 .env 파일이 있는지 확인\n"
    "      2) OPENAI_API_KEY=sk-proj-... 형식인지 확인\n"
    "      📖 강의 모듈 1-4 참조"
)

# LangSmith 설정 상태 출력
tracing = os.getenv("LANGCHAIN_TRACING_V2", "false")
ls_key  = os.getenv("LANGCHAIN_API_KEY", "")

print("✅ 환경 준비 완료")
print(f"   OPENAI_API_KEY 앞 7자 : {os.getenv('OPENAI_API_KEY')[:7]}")
print(f"   LangSmith 추적        : {'ON ✅' if tracing == 'true' else 'OFF ⚠️  — LANGCHAIN_TRACING_V2=true 확인 (📖 모듈 1-5)'}")
print(f"   LangSmith 키          : {'있음 ✅' if ls_key else '없음 ⚠️  — 📖 모듈 1-5 참조'}")

✅ 환경 준비 완료
   OPENAI_API_KEY 앞 7자 : sk-proj
   LangSmith 추적        : ON ✅
   LangSmith 키          : 있음 ✅


## 📋 모듈 1-1 ~ 1-5 체크리스트 (터미널 작업)

> ℹ️ 모듈 1-1 ~ 1-5는 VS Code 터미널에서 직접 수행합니다.
> 아래 항목을 모두 완료한 뒤 이 노트북의 실습을 진행하세요.
> 셀 2 환경 점검이 `✅ 환경 준비 완료`를 출력하면 OK입니다.

---

### ✅ 모듈 1-1 · VS Code 확장 설치
- [ ] **Python** 확장 설치 (게시자: Microsoft)
- [ ] **Jupyter** 확장 설치 (게시자: Microsoft)
- [ ] `hello.py` 실행 → 터미널에 "안녕하세요..." 출력 확인

### ✅ 모듈 1-2 · 파이썬 가상환경
- [ ] `python -m venv .venv` 실행 완료
- [ ] 활성화 후 터미널 프롬프트에 `(.venv)` 표시 확인
- [ ] VS Code 인터프리터를 `.venv`로 선택 완료
  (`Ctrl+Shift+P` → "Python: Select Interpreter")
- [ ] `langchain`, `langchain-openai`, `python-dotenv`, `langsmith` 설치 완료

### ✅ 모듈 1-3 · 터미널 기초
- [ ] `pwd` / `ls` / `cd` / `python` / `pip` 5개 명령 직접 실행 완료

### ✅ 모듈 1-4 · API 키 관리
- [ ] `.env` 파일 생성 (`OPENAI_API_KEY=sk-proj-...` 포함)
- [ ] `.gitignore`에 `.env` 한 줄 추가 완료
- [ ] OpenAI Usage Limit 설정 완료 (월 10달러 이상 권장)

### ✅ 모듈 1-5 · LangSmith 연결
- [ ] smith.langchain.com 가입 완료
- [ ] `.env`에 `LANGCHAIN_API_KEY`, `LANGCHAIN_PROJECT`, `LANGCHAIN_TRACING_V2=true` 추가 완료

---

**→ 위 항목 완료 후 셀 2 (환경 점검)를 실행하세요**

## Step 1. 랭체인 첫 호출 — v1 (문자열로 직접)

📖 **강의 연계**: 모듈 1-6 → "v1: 가장 단순한 형태"

LLM을 호출하는 가장 기본적인 형태입니다.
문자열을 `invoke()`에 바로 넣으면 `AIMessage` 객체가 반환됩니다.

아래 셀을 실행하면:
- `response.content` → AI의 실제 텍스트 답변
- `response.usage_metadata` → 토큰 수 (비용 계산 가능)
- LangSmith 대시보드에 첫 트레이스가 자동 생성됩니다 🎉

In [5]:
# Step 1-① 그대로 실행 — 강의 코드 그대로 실행해보기
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# temperature=0 : 일관된 답변 / temperature=1 : 창의적 답변

response = llm.invoke("LG CNS의 주요 사업을 3줄로 요약해줘")

print("=== AI 응답 ===")
print(response.content)
# 예상 출력: LG CNS는 IT 서비스, 클라우드, 스마트물류 분야를 주력 사업으로 합니다. ...

print("\n=== 토큰 사용량 ===")
print(response.usage_metadata)
# 예상 출력: {'input_tokens': ~22, 'output_tokens': ~80, 'total_tokens': ~102}
# gpt-4o-mini 기준 이번 호출 비용 ≈ $0.00006 (약 0.09원)

=== AI 응답 ===
LG CNS는 IT 서비스 및 솔루션 제공업체로, 클라우드, 인공지능, 빅데이터 분석 등 다양한 디지털 전환 서비스를 제공합니다. 또한, 스마트 팩토리와 IoT(사물인터넷) 솔루션을 통해 제조업의 효율성을 높이는 데 기여하고 있습니다. 공공, 금융, 제조 등 여러 산업 분야에 걸쳐 맞춤형 IT 솔루션을 제공하여 고객의 비즈니스 혁신을 지원합니다.

=== 토큰 사용량 ===
{'input_tokens': 21, 'output_tokens': 105, 'total_tokens': 126, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [ ]:
llm #다양한 파라미터들 조절 가능

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027C5448ACF0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000027C5448B8C0>, root_client=<openai.OpenAI object at 0x0000027C53741E80>

In [ ]:
response #메인은 contents이지만, 다양한 파라미터들을 조절할 수 있다는 것을 확인 가능

AIMessage(content='LG CNS는 IT 서비스 및 솔루션 제공업체로, 클라우드, 빅데이터, 인공지능(AI) 등 첨단 기술을 활용한 디지털 전환 서비스를 제공합니다. 또한, 스마트 팩토리와 같은 산업 자동화 솔루션을 통해 제조업의 효율성을 높이는 데 기여하고 있습니다. 공공, 금융, 물류 등 다양한 산업 분야에 맞춤형 IT 서비스를 제공하여 고객의 비즈니스 혁신을 지원합니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 105, 'prompt_tokens': 21, 'total_tokens': 126, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_75a1a369d7', 'id': 'chatcmpl-ELOeRviX5LVi66lLDwDabyemUplaZ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a07ae6-fe09-7633-a040-d4a21bd43116-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_t

In [13]:
# Step 1-② 한 곳만 바꾸기 — temperature를 바꿔서 차이 관찰
# TODO(🔰): temperature 값을 바꿔서 창의적인 답변이 나오는지 확인하세요
#            현재값 0.0 (일관된 답변) → 0.7 또는 1.0 으로 바꿔보세요
#            (힌트: 📖 강의 모듈 1-6 "temperature: 0=일관, 1=창의적")
temperature_test = 0.0  # ← 이 값을 바꿔서 실험!

llm_test = ChatOpenAI(model="gpt-4o-mini", temperature=temperature_test)
response_test = llm_test.invoke("강남역 주변 맛집 3개를 추천해줘.")

print(f"temperature={temperature_test} 결과:")
print(response_test.content)
# 위 Step 1-①(temperature=0)의 답변과 비교해보세요 — 어떤 차이가 있나요?
# LangSmith에서도 두 트레이스를 나란히 비교해보세요

temperature=0.0 결과:
강남역 주변에는 맛집이 많이 있습니다. 그 중에서 추천할 만한 3곳을 소개해드릴게요.

1. **부대찌개 1번가**  
   - 부대찌개 전문점으로, 진한 국물과 다양한 재료가 어우러진 부대찌개를 맛볼 수 있습니다. 매운맛 조절이 가능해 취향에 맞게 즐길 수 있습니다.

2. **마포갈매기**  
   - 고기와 함께 다양한 반찬이 제공되는 갈매기살 전문점입니다. 신선한 고기를 숯불에 구워 먹는 맛이 일품이며, 분위기도 좋습니다.

3. **이태원 부대찌개**  
   - 이곳은 부대찌개 외에도 다양한 한식 메뉴를 제공합니다. 특히, 푸짐한 양과 깊은 맛이 특징이며, 친구들과 함께 가기 좋은 곳입니다.

이 외에도 강남역 주변에는 다양한 맛집이 많으니, 여러 곳을 시도해보시는 것도 좋습니다!


In [14]:
# Step 1-② 한 곳만 바꾸기 — temperature를 바꿔서 차이 관찰
# TODO(🔰): temperature 값을 바꿔서 창의적인 답변이 나오는지 확인하세요
#            현재값 0.0 (일관된 답변) → 0.7 또는 1.0 으로 바꿔보세요
#            (힌트: 📖 강의 모듈 1-6 "temperature: 0=일관, 1=창의적")
temperature_test = 0.5  # ← 이 값을 바꿔서 실험!

llm_test = ChatOpenAI(model="gpt-4o-mini", temperature=temperature_test)
response_test = llm_test.invoke("강남역 주변 맛집 3개를 추천해줘.")

print(f"temperature={temperature_test} 결과:")
print(response_test.content)
# 위 Step 1-①(temperature=0)의 답변과 비교해보세요 — 어떤 차이가 있나요?
# LangSmith에서도 두 트레이스를 나란히 비교해보세요

temperature=0.5 결과:
강남역 주변에는 다양한 맛집이 많습니다. 여기 세 곳을 추천해드릴게요.

1. **육회자매집** - 신선한 육회로 유명한 이곳은 고소한 육회와 함께 다양한 반찬이 제공됩니다. 고기 본연의 맛을 느낄 수 있는 메뉴가 많아 육회 애호가들에게 인기가 많습니다.

2. **미정국수0410** - 이곳은 국수 전문점으로, 다양한 종류의 면 요리를 제공합니다. 특히 비빔국수와 칼국수가 인기 있으며, 면발이 쫄깃하고 소스가 맛있어 많은 사람들이 찾는 맛집입니다.

3. **청기와대** - 전통 한식을 제공하는 이곳은 다양한 한정식 메뉴가 있어 가족 모임이나 특별한 날에 적합합니다. 정갈한 반찬과 함께 나오는 메인 요리들이 인상적입니다.

강남역 주변은 맛집이 많으니, 다양한 음식을 즐겨보세요!


## Step 2. v2 — System + Human 메시지 분리

📖 **강의 연계**: 모듈 1-6 → "v2: 역할 분리" + "v1 vs v2 차이 비교표"

v1이 문자열을 바로 넣었다면, v2는 **역할(System)**과 **업무 지시(Human)**를 분리합니다.

| 메시지 종류 | 역할 | 예시 |
|-----------|------|------|
| `SystemMessage` | AI의 직책·행동 지침 (자주 바꾸지 않음) | "당신은 IT 전문가입니다" |
| `HumanMessage` | 실제 업무 지시 (매번 바뀌는 입력) | "LG CNS 요약해줘" |

> 💡 이 분리 구조가 내일 배울 **PromptTemplate**의 기초가 됩니다.
> LangSmith → 이 트레이스의 Inputs 탭에서 SystemMessage가 따로 보입니다.

In [5]:
# Step 2-① 그대로 실행 — SystemMessage + HumanMessage 분리
from langchain_core.messages import SystemMessage, HumanMessage

response_v2 = llm.invoke([
    SystemMessage(content="당신은 IT 기업 분석 전문가입니다. 한국어로 간결하게 답하세요."),
    HumanMessage(content="LG CNS의 주요 사업을 3줄로 요약해줘"),
])

print("=== v2 응답 (System 역할 적용) ===")
print(response_v2.content)
# v1(Step 1)과 답변이 어떻게 다른지 비교해보세요
# 📊 LangSmith Inputs 탭 → SystemMessage와 HumanMessage가 분리 저장된 것 확인

=== v2 응답 (System 역할 적용) ===
LG CNS는 IT 서비스 및 솔루션 제공업체로, 클라우드, 빅데이터, 인공지능(AI) 등 디지털 전환을 지원합니다. 또한, 스마트 팩토리와 IoT(사물인터넷) 솔루션을 통해 제조업의 효율성을 높이는 데 기여하고 있습니다. 공공 및 금융 분야에서도 다양한 IT 시스템 구축 및 운영 서비스를 제공합니다.


In [6]:
# Step 2-② 한 곳만 바꾸기 — System 역할만 바꿔서 답변 변화 관찰
# TODO(🔰): system_content 값을 다른 전문가 역할로 바꿔보세요
#            (예: "마케팅 전문가", "경력 10년 투자 분석가", "쉽게 설명하는 선생님")
#            (힌트: 역할이 바뀌면 같은 질문에도 다른 관점의 답변이 나옵니다)
system_content = "당신은 IT 기업 분석 전문가입니다. 영어로 간결하게 답하세요."  # ← 이 부분을 바꾸세요!

response_custom = llm.invoke([
    SystemMessage(content=system_content),
    HumanMessage(content="LG CNS의 주요 사업을 3줄로 요약해줘"),
])

print(f"적용한 System 역할: {system_content[:40]}...")
print("\n결과:")
print(response_custom.content)
# 역할이 바뀌면 같은 질문에도 다른 관점의 답변이 나오나요?

적용한 System 역할: 당신은 IT 기업 분석 전문가입니다. 영어로 간결하게 답하세요....

결과:
LG CNS specializes in IT services, including system integration, cloud computing, and digital transformation solutions. The company focuses on various industries such as manufacturing, finance, and public services. Additionally, LG CNS is investing in emerging technologies like AI and IoT to enhance its service offerings.


## 🔰 기본 미션 — 마이 서비스 조각 첫 호출

📖 **강의 연계**: 모듈 1-7 → "마이 서비스 첫 코드 작성" + "파일명 버전 관리 안내"

Day 1 슬랙에서 선언한 **마이 서비스 조각**에 맞게 아래 코드를 수정하고 실행하세요.

> 오늘 코드가 `my_service_v1.py`의 핵심이 됩니다
> - Day 2 PromptTemplate 적용 → `my_service_v2.py`
> - Day 3 Pydantic 적용 → `my_service_v3.py`
> - Day 4 Function Calling 적용 → `my_service_v4.py`

| 서비스 아이디어 예시 | system_prompt 예시 |
|------------------|------------------|
| 회의록 요약기 | "당신은 회의록 요약 전문가입니다. 결정사항·액션아이템을 구조화해 정리하세요." |
| 이메일 초안 도우미 | "당신은 비즈니스 이메일 작성 전문가입니다. 정중하고 명확한 한국어로 작성하세요." |
| 민원 분류기 | "고객 문의를 [배송/환불/제품불량/기타] 중 하나로 분류하고 이유를 한 줄로 설명하세요." |
| 사내 FAQ 봇 | "당신은 회사 정책 안내 전문가입니다. 질문에 대해 간결하고 정확하게 답하세요." |

**📤 제출**: 이 셀 실행 후 LangSmith 트레이스 링크를 `#day1-실습` 슬랙 채널에 제출

In [8]:
# 🔰 기본 미션: 아래 두 줄을 내 서비스에 맞게 수정하세요!
# TODO(🔰-1): AI 역할을 내 서비스에 맞게 수정하세요
#             (힌트: 📖 강의 모듈 1-7 "마이 서비스 조각 예시" 표 참고)
system_prompt = "당신은 게임 전문가입니다. 게임 용어를 활용하여 구체적으로 답하세요."  # ← 내 서비스에 맞게 수정!

# TODO(🔰-2): 내 서비스에 실제로 넣을 텍스트로 수정하세요
user_input = "던그리드에 대해 설명해줘."  # ← 실제 서비스 입력 예시로 수정!

response_my = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_input),
])

print("=== 내 서비스 첫 출력 ===")
print(response_my.content)
print(f"\n📊 토큰 사용: {response_my.usage_metadata}")
# 실행 후 LangSmith 트레이스 링크를 복사해 슬랙 #day1-실습 에 제출하세요!

=== 내 서비스 첫 출력 ===
던그리드(Dungeon Grid)는 주로 턴제 RPG나 던전 크롤러 게임에서 사용되는 맵 구성 방식으로, 플레이어가 탐험할 던전의 구조를 격자 형태로 나누어 표현하는 시스템입니다. 이 시스템은 다음과 같은 특징을 가지고 있습니다:

1. **격자 기반 탐험**: 던전은 정사각형 또는 육각형의 격자로 나뉘어 있으며, 각 격자는 플레이어가 이동할 수 있는 공간을 나타냅니다. 플레이어는 턴마다 한 칸씩 이동하며, 이를 통해 던전의 다양한 구역을 탐험할 수 있습니다.

2. **전투 및 전략**: 던그리드는 전투 시스템과 밀접하게 연결되어 있습니다. 플레이어는 적과의 거리를 고려하여 전략적으로 이동하고 공격할 수 있으며, 장애물이나 함정 등을 활용하여 전투를 유리하게 이끌 수 있습니다.

3. **탐험과 발견**: 격자 형태의 맵은 숨겨진 방이나 보물, 적의 위치 등을 쉽게 배치할 수 있게 해줍니다. 플레이어는 각 격자를 탐험하면서 새로운 아이템이나 퀘스트를 발견할 수 있습니다.

4. **레벨 디자인**: 던그리드는 레벨 디자인에 유용하며, 개발자는 다양한 형태의 던전을 쉽게 구성할 수 있습니다. 복잡한 퍼즐이나 다양한 경로를 통해 플레이어에게 도전과제를 제공할 수 있습니다.

5. **시각적 표현**: 던그리드는 종종 2D 또는 3D 그래픽으로 표현되며, 각 격자는 다양한 환경 요소(예: 벽, 문, 아이템 등)로 채워져 플레이어의 몰입감을 높입니다.

던그리드는 이러한 요소들 덕분에 많은 RPG와 던전 크롤러 게임에서 인기 있는 맵 구성 방식으로 자리잡고 있습니다. 대표적인 게임으로는 '던전 앤 드래곤' 시리즈나 '다크 소울' 시리즈의 일부 요소들이 있습니다.

📊 토큰 사용: {'input_tokens': 40, 'output_tokens': 459, 'total_tokens': 499, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

## ⭐ 심화 미션 — GPT-4o-mini vs Claude Haiku 비교 (선택)

📖 **강의 연계**: 모듈 1-6 → "⭐ 심화: ChatAnthropic으로 비교"

**사전 준비**: `.env`에 `ANTHROPIC_API_KEY=sk-ant-...` 추가 필요
(`pip install langchain-anthropic` 도 필요합니다)

동일한 질문을 두 모델에 보내고 결과를 비교하세요.

**목표**
- 두 모델의 응답 스타일·길이 차이 관찰
- LangSmith에서 비용(토큰 수)과 지연시간(ms) 비교
- 랭체인 비유 ②("Spring/Django 같은 프레임워크") 체감 — 모델이 달라도 `.invoke()` 인터페이스는 동일

> 기본 미션을 완료한 수강생은 이 미션에 도전하세요

In [10]:
# ⭐ 심화 미션 — GPT-4o-mini vs Claude Haiku 비교
# 사전 준비: pip install langchain-anthropic
#            .env에 ANTHROPIC_API_KEY=sk-ant-... 추가

# TODO(⭐): 아래 주석을 해제하고 빈칸을 채워 완성하세요
#   1. ChatAnthropic import
#   2. question 리스트 정의 (SystemMessage + HumanMessage)
#   3. GPT-4o-mini와 Claude Haiku 각각 invoke
#   4. 두 결과 나란히 출력

from langchain_anthropic import ChatAnthropic

question = [
    SystemMessage(content="당신은 IT 기업 분석 전문가입니다."),
    HumanMessage(content="LG CNS의 주요 사업을 3줄로 요약해줘"),
]

llm_gpt = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response_gpt = llm_gpt.invoke(question)

claude = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)   # TODO: 모델명 채우기
response_claude = claude.invoke(question)

print("=== GPT-4o-mini ===")
print(response_gpt.content)
print("\n=== Claude Haiku ===")
print(response_claude.content)

print("⭐ 심화 미션: 위 주석을 해제하고 빈칸을 채워 완성하세요")
print("   완료 후 LangSmith에서 두 모델의 비용·지연시간을 비교해보세요")

TypeError: Anthropic authentication failed: no API key or authorization credentials were provided. Set the ANTHROPIC_API_KEY environment variable, pass api_key=... to ChatAnthropic, or provide credentials via default_headers={"Authorization": ...}. If you are routing through the LangSmith gateway, set LANGSMITH_GATEWAY and LANGSMITH_GATEWAY_API_KEY.

## 🎓 Colab 졸업 미션 — 노트북 코드를 `.py`로 이식

📖 **강의 연계**: 모듈 1-7 → "Colab 이식 실습" + "변환 대응표"

이 노트북의 Step 2 코드를 **`first_call.py` 파일로 직접 작성**하고 **터미널에서 실행**해보세요.

> 📌 **왜 이 미션이 중요한가?**
> 노트북은 탐구·실험용, `.py`는 실제 서비스용입니다.
> 파이프라인 모듈부터는 `uvicorn app.main:app --reload` 처럼 터미널에서 서버를 실행합니다.
> 지금 이 감각을 익혀두세요.

### `first_call.py` 작성 내용

```python
# first_call.py — 프로젝트 루트에 이 파일을 새로 만드세요
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

response = llm.invoke([
    SystemMessage(content="당신은 IT 기업 분석 전문가입니다. 한국어로 간결하게 답하세요."),
    HumanMessage(content="LG CNS의 주요 사업을 3줄로 요약해줘"),
])

print("=== AI 응답 ===")
print(response.content)
print("\n=== 토큰 사용량 ===")
print(response.usage_metadata)
```

### 터미널 실행 명령 (가상환경 활성화 상태에서)

```bash
python first_call.py
```

In [ ]:
# 🎓 Colab 졸업 미션 확인
# 터미널에서 first_call.py 를 실행한 결과를 아래 TODO 주석에 붙여넣으세요

# TODO(🔰): 터미널 실행 결과를 여기에 붙여넣으세요
# ─── 아래에 붙여넣기 ───────────────────────
# === AI 응답 ===
# (여기에 실제 출력 내용 붙여넣기)
#
# === 토큰 사용량 ===
# (여기에 실제 출력 내용 붙여넣기)
# ───────────────────────────────────────────

print("✅ 터미널 실행 결과를 위 TODO 주석에 붙여넣으면 Colab 졸업 미션 완료!")
print("   노트북 ≠ 서비스 파일, 이제 터미널에서 LLM을 호출할 수 있습니다 🎓")

## 📬 제출 & 자가 체크

### ✅ Day 1 실습 최종 체크포인트
- [ ] 셀 2 환경 점검이 `✅ 환경 준비 완료`를 출력한다
- [ ] Step 1-①: `llm.invoke()`로 LLM 응답을 받고 `response.content`를 출력했다
- [ ] Step 1-②: `temperature` 값을 바꿔서 답변 차이를 직접 관찰했다
- [ ] Step 2-①: `SystemMessage` / `HumanMessage`를 분리해 v2를 실행했다
- [ ] Step 2-②: System 역할을 바꿔서 답변이 달라지는 것을 확인했다
- [ ] 🔰 기본 미션: 마이 서비스 조각의 첫 호출 코드를 작성했다
- [ ] 🎓 Colab 졸업 미션: `first_call.py`를 터미널에서 실행하고 결과를 붙여넣었다
- [ ] LangSmith 대시보드에 오늘의 트레이스가 여러 개 보인다

### 📤 제출 방법
슬랙 `#day1-실습` 채널에 아래 형식으로 제출:
```
[이름] Day1 LangSmith 트레이스: https://smith.langchain.com/...
[이름] 마이 서비스 조각: [서비스명] — [한 줄 설명]
```

---

### ➡️ Day 2 예고 — 내일 배울 것
- 오늘 만든 `system_prompt`에 **RCIF 프레임워크**를 적용합니다
- 내 서비스의 첫 번째 **ChatPromptTemplate**을 만들 예정입니다
- 키워드: RCIF / Zero-shot vs Few-shot / 반복 개선 워크숍

**내일 실습을 위한 숙제**: 내 서비스 조각에서 프롬프트가 잘 안 되는 케이스 1개를 떠올려오세요